# 🎤 Microphone Test & Live Inference

Tests your laptop microphone and runs trained gunshot detection models on live audio.

## What This Notebook Does
1. **Diagnoses** your microphone setup (Windows permissions, device detection)
2. **Tests** recording and playback
3. **Measures** your noise floor
4. **Runs** your trained CNN model on live audio
5. **Monitors** continuously with overlapping windows (Ring Buffer)

## Before Running
- Train at least one model (1D or 2D CNN) first
- See `manual/README.md` if you get microphone errors

In [1]:
# ============================================================
# CELL 1: Install Dependencies
# ============================================================
%pip install -q sounddevice soundfile numpy librosa tensorflow matplotlib

Note: you may need to restart the kernel to use updated packages.


In [5]:
# ============================================================
# CELL 2: Imports & Configuration
# ============================================================
import os
import sys
import time
import json
import numpy as np
import sounddevice as sd
import soundfile as sf
import librosa
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
import warnings
import IPython.display as ipd

import tensorflow as tf
from tensorflow import keras

warnings.filterwarnings('ignore')

# ==========================================================
# CONFIGURATION
# ==========================================================
sd.default.device[0] = 5  # <--- Forces it to use Device 5


SAMPLE_RATE = 22050       # Must match your trained model
CLIP_DURATION_MS = 250
TARGET_SAMPLES = int(SAMPLE_RATE * CLIP_DURATION_MS / 1000)

# Model paths — update these after training
# Option 1: 1D CNN
MODEL_PATH = Path(r'..\01_1D_CNN\output\1d_cnn_best.h5')
MODEL_TYPE = '1d'         # '1d' or '2d'

# Option 2: 2D CNN (uncomment to use)
# MODEL_PATH = Path(r'..\02_2D_CNN\output\2d_cnn_mel_spectrogram_best.h5')
# MODEL_TYPE = '2d'

# Feature extraction for 2D CNN
FEATURE_TYPE = 'mel_spectrogram'  # 'mel_spectrogram' or 'mfcc'
N_MELS = 64
N_MFCC = 13
N_FFT = 512
HOP_LENGTH = 128

# Continuous monitoring
OVERLAP = 0.5             # 50% overlap between windows (Ring Buffer)
CONFIDENCE_THRESHOLD = 0.7

print('✅ Configuration loaded.')
print(f'   Sample Rate: {SAMPLE_RATE} Hz')
print(f'   Clip: {CLIP_DURATION_MS}ms ({TARGET_SAMPLES} samples)')
print(f'   Model: {MODEL_PATH}')

✅ Configuration loaded.
   Sample Rate: 22050 Hz
   Clip: 250ms (5512 samples)
   Model: ..\01_1D_CNN\output\1d_cnn_best.h5


In [6]:
# ============================================================
# CELL 3: Microphone Diagnostic
# ============================================================

print('🔍 MICROPHONE DIAGNOSTIC\n')
print('=' * 60)

# Step 1: List all audio devices
print('\n📋 DETECTED AUDIO DEVICES:')
print('-' * 60)
devices = sd.query_devices()
input_devices = []

for i, d in enumerate(devices):
    if d['max_input_channels'] > 0:
        input_devices.append(i)
        default_mark = ' ⭐ DEFAULT' if i == sd.default.device[0] else ''
        print(f'  [{i:2d}] {d["name"]}')
        print(f'       Inputs: {d["max_input_channels"]} ch | '
              f'Rate: {d["default_samplerate"]:.0f} Hz{default_mark}')

if not input_devices:
    print('\n❌ NO INPUT DEVICES FOUND!')
    print('\n🔧 FIX: Check Windows Settings:')
    print('   1. Open: Settings → Privacy → Microphone')
    print('   2. Turn ON: "Microphone access"')
    print('   3. Turn ON: "Let desktop apps access your microphone"')
    print('   4. RESTART your Jupyter kernel and this notebook')
else:
    print(f'\n✅ Found {len(input_devices)} input device(s)')

# Step 2: Test microphone access
print('\n📋 MICROPHONE ACCESS TEST:')
print('-' * 60)

try:
    # Try to record 0.1 seconds
    test_audio = sd.rec(int(0.1 * SAMPLE_RATE), samplerate=SAMPLE_RATE,
                        channels=1, dtype='float32')
    sd.wait()
    print('✅ Microphone access GRANTED')
    print(f'   Recorded {len(test_audio)} samples successfully')
    
    # Check if it's just silence (might be a fake/virtual device)
    rms = np.sqrt(np.mean(test_audio ** 2))
    if rms < 1e-8:
        print('⚠️ WARNING: Recording is pure silence. Device might be muted or virtual.')
    else:
        print(f'   Signal detected (RMS: {rms:.6f})')

except sd.PortAudioError as e:
    print(f'❌ MICROPHONE ACCESS DENIED: {e}')
    print(f'\n🔧 FIX (Windows):')
    print(f'   1. Open: Settings → Privacy & Security → Microphone')
    print(f'   2. Turn ON: "Microphone access"')
    print(f'   3. Scroll down → Turn ON: "Let desktop apps access your microphone"')
    print(f'   4. RESTART your Jupyter/VS Code and try again')

except Exception as e:
    print(f'❌ Unexpected error: {e}')

# Step 3: Sample rate support
print(f'\n📋 SAMPLE RATE COMPATIBILITY:')
print('-' * 60)
for rate in [16000, 22050, 44100, 48000]:
    try:
        sd.check_input_settings(samplerate=rate, channels=1)
        match = ' ← YOUR MODEL' if rate == SAMPLE_RATE else ''
        print(f'  ✅ {rate:>6} Hz supported{match}')
    except Exception:
        match = ' ← YOUR MODEL ⚠️ NOT SUPPORTED' if rate == SAMPLE_RATE else ''
        print(f'  ❌ {rate:>6} Hz NOT supported{match}')

print(f'\n{"=" * 60}')

🔍 MICROPHONE DIAGNOSTIC


📋 DETECTED AUDIO DEVICES:
------------------------------------------------------------
  [ 5] Microphone (Realtek HD Audio Mic input)
       Inputs: 2 ch | Rate: 44100 Hz ⭐ DEFAULT
  [ 6] Line In (Realtek HD Audio Line input)
       Inputs: 2 ch | Rate: 44100 Hz
  [ 7] Stereo Mix (Realtek HD Audio Stereo input)
       Inputs: 2 ch | Rate: 44100 Hz

✅ Found 3 input device(s)

📋 MICROPHONE ACCESS TEST:
------------------------------------------------------------
❌ MICROPHONE ACCESS DENIED: Error opening InputStream: Invalid device [PaErrorCode -9996]

🔧 FIX (Windows):
   1. Open: Settings → Privacy & Security → Microphone
   2. Turn ON: "Microphone access"
   3. Scroll down → Turn ON: "Let desktop apps access your microphone"
   4. RESTART your Jupyter/VS Code and try again

📋 SAMPLE RATE COMPATIBILITY:
------------------------------------------------------------
  ❌  16000 Hz NOT supported
  ❌  22050 Hz NOT supported ← YOUR MODEL ⚠️ NOT SUPPORTED
  ✅  44100 Hz 

In [7]:
# ============================================================
# CELL 4: Record & Playback Test
# ============================================================

RECORD_SECONDS = 3

print(f'🎤 Recording {RECORD_SECONDS} seconds... (make some noise!)')
recording = sd.rec(int(RECORD_SECONDS * SAMPLE_RATE),
                   samplerate=SAMPLE_RATE, channels=1, dtype='float32')
sd.wait()
recording = recording.flatten()
print(f'✅ Recorded {len(recording):,} samples')

# Waveform plot
fig, axes = plt.subplots(2, 1, figsize=(14, 6))

t = np.arange(len(recording)) / SAMPLE_RATE
axes[0].plot(t, recording, linewidth=0.5, color='#3498db')
axes[0].set_title('Recorded Waveform', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Amplitude')

# Spectrogram
S = librosa.feature.melspectrogram(y=recording, sr=SAMPLE_RATE, n_mels=64)
S_dB = librosa.power_to_db(S, ref=np.max)
librosa.display.specshow(S_dB, sr=SAMPLE_RATE, x_axis='time', y_axis='mel', ax=axes[1])
axes[1].set_title('Mel Spectrogram', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# Playback
print('\n🔊 Playback:')
display(ipd.Audio(recording, rate=SAMPLE_RATE))

# Stats
rms = np.sqrt(np.mean(recording ** 2))
peak = np.max(np.abs(recording))
print(f'\n📊 Recording stats:')
print(f'   RMS: {rms:.6f}')
print(f'   Peak: {peak:.6f}')
print(f'   Dynamic range: {20*np.log10(peak/max(rms, 1e-10)):.1f} dB')

🎤 Recording 3 seconds... (make some noise!)


PortAudioError: Error opening InputStream: Invalid device [PaErrorCode -9996]

In [ ]:
# ============================================================
# CELL 5: Noise Floor Analysis
# ============================================================

SILENCE_SECONDS = 5

print(f'🤫 Recording {SILENCE_SECONDS}s of SILENCE...')
print(f'   (Stay quiet! This measures your background noise floor)\n')

silence = sd.rec(int(SILENCE_SECONDS * SAMPLE_RATE),
                 samplerate=SAMPLE_RATE, channels=1, dtype='float32')
sd.wait()
silence = silence.flatten()

# Analyze noise floor
noise_rms = np.sqrt(np.mean(silence ** 2))
noise_peak = np.max(np.abs(silence))
noise_db = 20 * np.log10(max(noise_rms, 1e-10))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Noise waveform
t = np.arange(len(silence)) / SAMPLE_RATE
axes[0].plot(t, silence, linewidth=0.3, color='#95a5a6', alpha=0.8)
axes[0].set_title('Noise Floor Waveform', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylim(-0.01, 0.01)

# Noise spectrum
S_noise = np.abs(librosa.stft(silence, n_fft=N_FFT))
noise_spectrum = np.mean(S_noise, axis=1)
freqs = librosa.fft_frequencies(sr=SAMPLE_RATE, n_fft=N_FFT)
axes[1].plot(freqs, 20*np.log10(noise_spectrum + 1e-10), color='#e74c3c')
axes[1].set_title('Noise Spectrum', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_ylabel('Magnitude (dB)')

plt.tight_layout()
plt.show()

print(f'\n📊 Noise Floor Analysis:')
print(f'   RMS: {noise_rms:.8f}')
print(f'   Peak: {noise_peak:.8f}')
print(f'   Level: {noise_db:.1f} dBFS')

if noise_db > -30:
    print(f'   ⚠️ Noise floor is HIGH. Background noise may affect detection.')
elif noise_db > -50:
    print(f'   ✅ Noise floor is acceptable.')
else:
    print(f'   ✅ Noise floor is LOW — excellent for detection.')

In [ ]:
# ============================================================
# CELL 6: Load Trained Model
# ============================================================

model_path = Path(MODEL_PATH)

if not model_path.exists():
    # Try resolving relative to notebook location
    model_path = Path.cwd().resolve() / MODEL_PATH

if not model_path.exists():
    print(f'❌ Model not found: {MODEL_PATH}')
    print(f'   Please train a model first (01_1D_CNN or 02_2D_CNN notebook)')
    print(f'   Then update MODEL_PATH in Cell 2')
else:
    model = keras.models.load_model(str(model_path))
    print(f'✅ Loaded model: {model_path.name}')
    print(f'   Type: {MODEL_TYPE.upper()} CNN')
    print(f'   Input shape: {model.input_shape}')
    print(f'   Parameters: {model.count_params():,}')
    
    # Load normalization stats for 2D CNN
    norm_stats = None
    if MODEL_TYPE == '2d':
        norm_path = model_path.parent / '2d_cnn_norm_stats.json'
        if norm_path.exists():
            norm_stats = json.loads(norm_path.read_text())
            print(f'   Norm stats loaded: mean={norm_stats["mean"]:.4f}, std={norm_stats["std"]:.4f}')
        else:
            print(f'   ⚠️ No norm stats found. Will use default normalization.')

In [ ]:
# ============================================================
# CELL 7: Single-Shot Prediction
# ============================================================

def preprocess_for_model(audio, sr, model_type, feature_type='mel_spectrogram',
                         n_mels=64, n_mfcc=13, n_fft=512, hop_length=128,
                         target_samples=None, norm_stats=None):
    """Preprocess a raw audio clip for model inference."""
    y = audio.copy().flatten()
    
    # Force exact length
    if target_samples:
        if len(y) >= target_samples:
            y = y[:target_samples]
        else:
            y = np.pad(y, (0, target_samples - len(y)))
    
    # Normalize
    peak = np.max(np.abs(y))
    if peak > 1e-6:
        y = y / peak
    
    if model_type == '1d':
        # Raw waveform: (1, samples, 1)
        return y.reshape(1, -1, 1).astype(np.float32)
    
    elif model_type == '2d':
        # Extract spectrogram features
        if feature_type == 'mel_spectrogram':
            S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels,
                                              n_fft=n_fft, hop_length=hop_length)
            feat = librosa.power_to_db(S, ref=np.max)
        elif feature_type == 'mfcc':
            feat = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc,
                                       n_fft=n_fft, hop_length=hop_length)
        
        # Normalize using training stats
        if norm_stats:
            feat = (feat - norm_stats['mean']) / max(norm_stats['std'], 1e-6)
        
        return feat.reshape(1, feat.shape[0], feat.shape[1], 1).astype(np.float32)


# --- Record a single clip ---
print(f'🎤 Recording {CLIP_DURATION_MS}ms clip...\n')
clip = sd.rec(TARGET_SAMPLES, samplerate=SAMPLE_RATE,
              channels=1, dtype='float32')
sd.wait()
clip = clip.flatten()

# Preprocess
x = preprocess_for_model(clip, SAMPLE_RATE, MODEL_TYPE, FEATURE_TYPE,
                         N_MELS, N_MFCC, N_FFT, HOP_LENGTH,
                         TARGET_SAMPLES, norm_stats)

# Predict
prob = model.predict(x, verbose=0).flatten()[0]
label = 'GUNSHOT 🔫' if prob >= 0.5 else 'NON-GUNSHOT 🎵'

# Display
color = '#e74c3c' if prob >= 0.5 else '#2ecc71'
print(f'\n{"=" * 40}')
print(f'  Prediction: {label}')
print(f'  Confidence: {prob:.4f}')
print(f'{"=" * 40}')

# Waveform
fig, ax = plt.subplots(figsize=(12, 3))
t = np.arange(len(clip)) / SAMPLE_RATE * 1000
ax.plot(t, clip, linewidth=0.8, color=color)
ax.set_title(f'{label} — Confidence: {prob:.4f}', fontsize=14, fontweight='bold', color=color)
ax.set_xlabel('Time (ms)')
plt.tight_layout()
plt.show()

display(ipd.Audio(clip, rate=SAMPLE_RATE))

In [ ]:
# ============================================================
# CELL 8: Continuous Monitoring (Ring Buffer)
# ============================================================
#
# This solves the "Guillotine Effect" — a gunshot might fall on
# the boundary between two buffers. By overlapping windows at
# 50%, the gunshot is always fully captured in at least one window.
# ============================================================

MONITOR_SECONDS = 30    # How long to monitor (change as needed)
WINDOW_OVERLAP = 0.5    # 50% overlap

hop_samples = int(TARGET_SAMPLES * (1 - WINDOW_OVERLAP))
n_windows = int((MONITOR_SECONDS * SAMPLE_RATE) / hop_samples)

print(f'🎤 CONTINUOUS MONITORING')
print(f'   Duration: {MONITOR_SECONDS}s')
print(f'   Window: {CLIP_DURATION_MS}ms with {int(WINDOW_OVERLAP*100)}% overlap')
print(f'   Threshold: {CONFIDENCE_THRESHOLD}')
print(f'   Total windows: {n_windows}')
print(f'\n🔴 LISTENING... (Ctrl+C or wait {MONITOR_SECONDS}s to stop)\n')
print(f'{"Time":<10} {"Prediction":<15} {"Confidence":<12} {""}')
print(f'{"-"*50}')

# Record full audio
full_audio = sd.rec(int(MONITOR_SECONDS * SAMPLE_RATE),
                    samplerate=SAMPLE_RATE, channels=1, dtype='float32')
sd.wait()
full_audio = full_audio.flatten()

# Process in overlapping windows
detections = []
all_probs = []

for i in range(n_windows):
    start = i * hop_samples
    end = start + TARGET_SAMPLES
    if end > len(full_audio):
        break
    
    window = full_audio[start:end]
    x = preprocess_for_model(window, SAMPLE_RATE, MODEL_TYPE, FEATURE_TYPE,
                             N_MELS, N_MFCC, N_FFT, HOP_LENGTH,
                             TARGET_SAMPLES, norm_stats)
    
    prob = model.predict(x, verbose=0).flatten()[0]
    all_probs.append(prob)
    
    time_sec = start / SAMPLE_RATE
    
    if prob >= CONFIDENCE_THRESHOLD:
        detections.append({
            'time': time_sec,
            'confidence': float(prob),
            'window_idx': i
        })
        print(f'{time_sec:>7.2f}s   {"🔫 GUNSHOT":<15} {prob:<12.4f} ← DETECTED!')

# --- Results ---
print(f'\n{"=" * 50}')
print(f'MONITORING COMPLETE')
print(f'{"=" * 50}')
print(f'Duration: {MONITOR_SECONDS}s')
print(f'Windows analyzed: {len(all_probs)}')
print(f'Detections: {len(detections)}')

# Plot timeline
fig, ax = plt.subplots(figsize=(16, 4))
times = [i * hop_samples / SAMPLE_RATE for i in range(len(all_probs))]
colors = ['#e74c3c' if p >= CONFIDENCE_THRESHOLD else '#2ecc71' for p in all_probs]
ax.bar(times, all_probs, width=hop_samples/SAMPLE_RATE * 0.8,
       color=colors, alpha=0.7)
ax.axhline(y=CONFIDENCE_THRESHOLD, color='orange', linestyle='--',
           label=f'Threshold ({CONFIDENCE_THRESHOLD})', linewidth=2)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Gunshot Probability')
ax.set_title('Continuous Monitoring Timeline', fontsize=14, fontweight='bold')
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

# Save detections
output_dir = Path.cwd().resolve() / 'output'
output_dir.mkdir(exist_ok=True)
(output_dir / 'detections.json').write_text(json.dumps(detections, indent=2))
print(f'\n📊 Detections saved to: {output_dir / "detections.json"}')